In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk 


df_full = pd.read_csv('15k_pos_neg_ratio_set.csv')

In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import re
import time

class MetaBlocking:
    def __init__(self, pruning_algorithm='BLAST', feature_set=None):
        """
        Initialize the Meta-blocking system.
        
        Args:
            pruning_algorithm: One of 'BLAST' or 'RCNP'
            feature_set: List of features to use. If None, use the optimal set from the paper.
        """
        self.pruning_algorithm = pruning_algorithm
        
        # Set default feature sets based on the paper's findings
        if feature_set is None:
            if pruning_algorithm == 'BLAST':
                # CF-IBF, RACCB, RS, NRS as recommended in the paper
                self.feature_set = ['CF-IBF', 'RACCB', 'RS', 'NRS']
            else:  # RCNP
                # CF-IBF, RACCB, JS, LCP, WJS as recommended in the paper
                self.feature_set = ['CF-IBF', 'RACCB', 'JS', 'LCP', 'WJS']
        else:
            self.feature_set = feature_set
            
        self.blocks = {}
        self.entity_blocks = defaultdict(set)
        self.classifier = LogisticRegression(random_state=42)
        
    def create_token_blocks(self, data, attributes):
        """
        Create blocks using Token Blocking technique.
        
        Args:
            data: List of entity dictionaries
            attributes: List of attribute names to use for blocking
        """
        self.blocks = defaultdict(set)
        self.entity_blocks = defaultdict(set)
        
        for entity_id, entity in enumerate(data):
            tokens = set()
            for attr in attributes:
                if attr in entity and entity[attr]:
                    # Tokenize the attribute value
                    attr_tokens = re.findall(r'\w+', str(entity[attr]).lower())
                    tokens.update(attr_tokens)
            
            # Create blocks for each token
            for token in tokens:
                self.blocks[token].add(entity_id)
                self.entity_blocks[entity_id].add(token)
        
        # Remove blocks that are too large (Block Purging)
        entity_count = len(data)
        self.blocks = {k: v for k, v in self.blocks.items() 
                      if len(v) <= entity_count / 2}
        
        # Block Filtering: remove entities from their largest blocks
        for entity_id in range(len(data)):
            if entity_id in self.entity_blocks:
                block_sizes = [(block, len(self.blocks[block])) 
                              for block in self.entity_blocks[entity_id] 
                              if block in self.blocks]
                
                if block_sizes:
                    # Sort by block size (descending)
                    block_sizes.sort(key=lambda x: x[1], reverse=True)
                    
                    # Remove entity from its largest 20% blocks
                    blocks_to_remove = block_sizes[:max(1, int(len(block_sizes) * 0.2))]
                    
                    for block, _ in blocks_to_remove:
                        if entity_id in self.blocks[block]:
                            self.blocks[block].remove(entity_id)
                            self.entity_blocks[entity_id].remove(block)
                            
        # Cleanup empty blocks
        self.blocks = {k: v for k, v in self.blocks.items() if v}
        
        return self.blocks, self.entity_blocks

In [4]:
def get_candidate_pairs(self):
        """Extract all candidate pairs from blocks"""
        candidate_pairs = set()
        
        for block, entities in self.blocks.items():
            entities = list(entities)
            for i in range(len(entities)):
                for j in range(i+1, len(entities)):
                    candidate_pairs.add((min(entities[i], entities[j]), 
                                         max(entities[i], entities[j])))
        
        return list(candidate_pairs)
    
def calculate_features(self, candidate_pairs, data):
    """
    Calculate features for each candidate pair.
    
    Args:
        candidate_pairs: List of entity pairs (i, j)
        data: List of entity dictionaries
    
    Returns:
        DataFrame with features for each candidate pair
    """
    # Create a DataFrame to store features
    features = []
    
    for i, j in candidate_pairs:
        # Common blocks
        common_blocks = self.entity_blocks[i].intersection(self.entity_blocks[j])
        num_common_blocks = len(common_blocks)
        
        # Skip if no common blocks
        if num_common_blocks == 0:
            continue
            
        feature_dict = {'entity_i': i, 'entity_j': j}
        
        # Calculate features based on the feature set
        if 'CF-IBF' in self.feature_set:
            # Co-occurrence Frequency-Inverse Block Frequency
            cf_ibf = num_common_blocks * np.log(len(self.blocks) / len(self.entity_blocks[i])) * np.log(len(self.blocks) / len(self.entity_blocks[j]))
            feature_dict['CF-IBF'] = cf_ibf
            
        if 'RACCB' in self.feature_set:
            # Reciprocal Aggregate Cardinality of Common Blocks
            raccb = sum(1 / sum(1 for pair in self.get_block_pairs(block) if pair[0] != pair[1]) 
                        for block in common_blocks if block in self.blocks)
            feature_dict['RACCB'] = raccb
            
        if 'JS' in self.feature_set:
            # Jaccard Scheme
            js = num_common_blocks / (len(self.entity_blocks[i]) + len(self.entity_blocks[j]) - num_common_blocks)
            feature_dict['JS'] = js
            
        if 'LCP' in self.feature_set:
            # Local Candidate Pairs
            lcp_i = sum(1 for e in data if any(e in self.blocks[block] for block in self.entity_blocks[i]))
            lcp_j = sum(1 for e in data if any(e in self.blocks[block] for block in self.entity_blocks[j]))
            feature_dict['LCP_i'] = lcp_i
            feature_dict['LCP_j'] = lcp_j
            
        if 'EJS' in self.feature_set:
            # Enhanced Jaccard Scheme
            total_blocks = sum(len(self.blocks[b]) for b in self.blocks)
            total_i = sum(len(self.blocks[b]) for b in self.entity_blocks[i] if b in self.blocks)
            total_j = sum(len(self.blocks[b]) for b in self.entity_blocks[j] if b in self.blocks)
            
            ejs = js * np.log(total_blocks / total_i) * np.log(total_blocks / total_j)
            feature_dict['EJS'] = ejs
            
        if 'RS' in self.feature_set:
            # Reciprocal Sizes Scheme
            rs = sum(1 / len(self.blocks[block]) for block in common_blocks if block in self.blocks)
            feature_dict['RS'] = rs
            
        if 'NRS' in self.feature_set:
            # Normalized Reciprocal Sizes Scheme
            sum_common = sum(1 / len(self.blocks[block]) for block in common_blocks if block in self.blocks)
            sum_i = sum(1 / len(self.blocks[block]) for block in self.entity_blocks[i] if block in self.blocks)
            sum_j = sum(1 / len(self.blocks[block]) for block in self.entity_blocks[j] if block in self.blocks)
            
            nrs = sum_common / (sum_i + sum_j - sum_common)
            feature_dict['NRS'] = nrs
            
        if 'WJS' in self.feature_set:
            # Weighted Jaccard Scheme
            sum_common = sum(1 / sum(1 for pair in self.get_block_pairs(block) if pair[0] != pair[1])
                            for block in common_blocks if block in self.blocks)
            
            sum_i = sum(1 / sum(1 for pair in self.get_block_pairs(block) if pair[0] != pair[1])
                        for block in self.entity_blocks[i] if block in self.blocks)
            
            sum_j = sum(1 / sum(1 for pair in self.get_block_pairs(block) if pair[0] != pair[1])
                        for block in self.entity_blocks[j] if block in self.blocks)
            
            wjs = sum_common / (sum_i + sum_j - sum_common)
            feature_dict['WJS'] = wjs
            
        features.append(feature_dict)
        
    return pd.DataFrame(features)

def get_block_pairs(self, block):
    """Get all pairs of entities in a block"""
    entities = list(self.blocks[block])
    pairs = []
    for i in range(len(entities)):
        for j in range(i, len(entities)):
            pairs.append((entities[i], entities[j]))
    return pairs

In [5]:
def train(self, labeled_data, features):
        """
        Train the model using labeled data.
        
        Args:
            labeled_data: DataFrame with columns 'entity_i', 'entity_j', 'label'
            features: DataFrame with features for candidate pairs
        """
        # Merge features with labels
        training_data = pd.merge(features, labeled_data, on=['entity_i', 'entity_j'], how='inner')
        
        # Select feature columns
        feature_cols = [col for col in training_data.columns 
                       if col in self.feature_set or (col.startswith('LCP_') and 'LCP' in self.feature_set)]
        
        X = training_data[feature_cols].values
        y = training_data['label'].values
        
        # Train the classifier
        self.classifier.fit(X, y)
        
        return self.classifier
    
def prune_pairs(self, features, data, r=0.35):
    """
    Apply pruning algorithm to candidate pairs.
    
    Args:
        features: DataFrame with features for candidate pairs
        data: Original entity data
        r: Pruning ratio for BLAST algorithm
        
    Returns:
        List of retained pairs
    """
    # Select feature columns
    feature_cols = [col for col in features.columns 
                    if col in self.feature_set or (col.startswith('LCP_') and 'LCP' in self.feature_set)]
    
    # Get probabilities
    X = features[feature_cols].values
    probabilities = self.classifier.predict_proba(X)[:, 1]  # Class 1 probabilities
    
    # Add probabilities to features dataframe
    features['probability'] = probabilities
    
    # Filter pairs with probability >= 0.5
    valid_pairs = features[features['probability'] >= 0.5].copy()
    
    retained_pairs = []
    
    if self.pruning_algorithm == 'BLAST':
        # BLAST algorithm
        max_probs = {}
        
        # First pass: find max probability for each entity
        for _, row in valid_pairs.iterrows():
            i, j = row['entity_i'], row['entity_j']
            prob = row['probability']
            
            max_probs[i] = max(max_probs.get(i, 0), prob)
            max_probs[j] = max(max_probs.get(j, 0), prob)
        
        # Second pass: apply pruning
        for _, row in valid_pairs.iterrows():
            i, j = row['entity_i'], row['entity_j']
            prob = row['probability']
            
            if prob >= r * (max_probs[i] + max_probs[j]):
                retained_pairs.append((i, j, prob))
                
    elif self.pruning_algorithm == 'RCNP':
        # RCNP algorithm
        # Calculate the value of k
        block_sizes = [len(entities) for entities in self.blocks.values()]
        k = max(1, int(sum(block_sizes) / len(data)))
        
        # Create priority queues for each entity
        queues = defaultdict(list)
        min_prob = defaultdict(float)
        
        # First pass: populate queues
        for _, row in valid_pairs.iterrows():
            i, j = row['entity_i'], row['entity_j']
            prob = row['probability']
            
            # For entity i
            if prob > min_prob[i]:
                queues[i].append((j, prob))
                queues[i].sort(key=lambda x: x[1], reverse=True)
                
                if len(queues[i]) > k:
                    _, min_prob[i] = queues[i].pop()
            
            # For entity j
            if prob > min_prob[j]:
                queues[j].append((i, prob))
                queues[j].sort(key=lambda x: x[1], reverse=True)
                
                if len(queues[j]) > k:
                    _, min_prob[j] = queues[j].pop()
        
        # Second pass: find reciprocal pairs
        for _, row in valid_pairs.iterrows():
            i, j = row['entity_i'], row['entity_j']
            prob = row['probability']
            
            # Check if j is in i's queue and i is in j's queue
            i_contains_j = any(e == j for e, _ in queues[i])
            j_contains_i = any(e == i for e, _ in queues[j])
            
            if i_contains_j and j_contains_i:
                retained_pairs.append((i, j, prob))
                
    return retained_pairs